# Actividad 4.4: Pipeline de Procesamiento

In [7]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_regression

# Cargamos datos
df = pd.DataFrame({
    'precio': [1200, 1500, 3000, 4500, 1000],
    'habitaciones': [1, 2, 3, 4, 1],
    'distancia_centro': [2.5, np.nan, 5.0, 1.2, 3.4], 
    'tipo': ['Privado', 'Compartido', 'Privado', 'Hotel', 'Privado'],
    'target_renta': [15000, 18000, 35000, 50000, 14000] 
})

In [8]:
# Definición de columnas por tipo
cols_num = ['precio', 'habitaciones', 'distancia_centro']
cols_cat = ['tipo']

# Diseño de los sub-pipelines (Transformación)
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler())                   
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), 
    ('onehot', OneHotEncoder(handle_unknown='ignore'))    
])

# Unión de transformaciones (ColumnTransformer)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, cols_num),
        ('cat', cat_transformer, cols_cat)
    ])

# Pipeline Final (Incluyendo Selección de Características)
final_pipeline = Pipeline(steps=[
    ('preprocesar', preprocessor),
    ('seleccion', SelectKBest(score_func=f_regression, k='all')) 
])

## Ejecución

In [9]:
X = df.drop('target_renta', axis=1)
y = df['target_renta']
datos_listos = final_pipeline.fit_transform(X, y)

columnas_finales = (cols_num + 
                   list(final_pipeline.named_steps['preprocesar']
                        .named_transformers_['cat']
                        .get_feature_names_out()))

df_resultado = pd.DataFrame(datos_listos, columns=columnas_finales)

print("--- Modelo pipeline ---")
print(df_resultado.head())

--- Modelo pipeline ---
     precio  habitaciones  distancia_centro  tipo_Compartido  tipo_Hotel  \
0 -0.781624     -1.028992         -0.412257              0.0         0.0   
1 -0.556155     -0.171499         -0.048501              1.0         0.0   
2  0.571186      0.685994          1.608609              0.0         0.0   
3  1.698528      1.543487         -1.463107              0.0         1.0   
4 -0.931936     -1.028992          0.315255              0.0         0.0   

   tipo_Privado  
0           1.0  
1           0.0  
2           1.0  
3           0.0  
4           1.0  
